Import Required Libraries

In [1]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

Create the Bell Pair (Charlie's Job)

In [2]:
def create_bell_pair():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    return qc

bell_circuit = create_bell_pair()
print("Bell pair circuit:")
print(bell_circuit)

Bell pair circuit:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
c: 2/══════════
               


Alice Encodes Her Message

In [3]:
def encode_message(qc, message):
    if message == '01':
        qc.z(0)
    elif message == '10':
        qc.x(0)
    elif message == '11':
        qc.z(0)
        qc.x(0)
    return qc

messages = ['00', '01', '10', '11']
encoded_circuits = []

for msg in messages:
    qc = create_bell_pair()
    qc = encode_message(qc, msg)
    encoded_circuits.append((msg, qc))
    print(f"Circuit for message {msg}:")
    print(qc)
    print()

Circuit for message 00:
     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘
c: 2/══════════
               

Circuit for message 01:
     ┌───┐     ┌───┐
q_0: ┤ H ├──■──┤ Z ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     
c: 2/═══════════════
                    

Circuit for message 10:
     ┌───┐     ┌───┐
q_0: ┤ H ├──■──┤ X ├
     └───┘┌─┴─┐└───┘
q_1: ─────┤ X ├─────
          └───┘     
c: 2/═══════════════
                    

Circuit for message 11:
     ┌───┐     ┌───┐┌───┐
q_0: ┤ H ├──■──┤ Z ├┤ X ├
     └───┘┌─┴─┐└───┘└───┘
q_1: ─────┤ X ├──────────
          └───┘          
c: 2/════════════════════
                         



Bob Decodes the Message

In [4]:
def decode_message(qc):
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc

decoded_circuits = []

for msg, qc in encoded_circuits:
    qc = decode_message(qc)
    decoded_circuits.append((msg, qc))
    print(f"Full circuit for message {msg}:")
    print(qc)
    print()

Full circuit for message 00:
     ┌───┐          ┌───┐┌─┐
q_0: ┤ H ├──■────■──┤ H ├┤M├
     └───┘┌─┴─┐┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ X ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 

Full circuit for message 01:
     ┌───┐     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├──■──┤ H ├┤M├
     └───┘┌─┴─┐└───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├─────┤ X ├─┤M├──╫─
          └───┘     └───┘ └╥┘  ║ 
c: 2/══════════════════════╩═══╩═
                           1   0 

Full circuit for message 10:
     ┌───┐     ┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ X ├──■──┤ H ├┤M├
     └───┘┌─┴─┐└───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├─────┤ X ├─┤M├──╫─
          └───┘     └───┘ └╥┘  ║ 
c: 2/══════════════════════╩═══╩═
                           1   0 

Full circuit for message 11:
     ┌───┐     ┌───┐┌───┐     ┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├┤ X ├──■──┤ H ├┤M├
     └───┘┌─┴─┐└───┘└───┘┌─┴─┐└┬─┬┘└╥┘
q_1: ─────┤ X ├──────────┤ X ├─┤M├──╫─
          └───┘          └───┘ └╥┘  ║ 
c: 2/══

Simulate and Verify

In [5]:
simulator = AerSimulator()

print("Message Sent → Message Received")
print("-" * 30)

for msg, qc in decoded_circuits:
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=1024).result()
    counts = result.get_counts()
    
    most_frequent = max(counts, key=counts.get)
    print(f"    {msg}       →       {most_frequent}")
    print(f"    Counts: {counts}\n")

Message Sent → Message Received
------------------------------
    00       →       00
    Counts: {'00': 1024}

    01       →       01
    Counts: {'01': 1024}

    10       →       10
    Counts: {'10': 1024}

    11       →       11
    Counts: {'11': 1024}



Complete Code

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit import transpile

def create_bell_pair():
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    return qc

def encode_message(qc, message):
    if message == '01':
        qc.z(0)
    elif message == '10':
        qc.x(0)
    elif message == '11':
        qc.z(0)
        qc.x(0)
    return qc

def decode_message(qc):
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc

simulator = AerSimulator()
messages = ['00', '01', '10', '11']

print("Message Sent → Message Received")
print("-" * 30)

for msg in messages:
    qc = create_bell_pair()
    qc = encode_message(qc, msg)
    qc = decode_message(qc)
    
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=1024).result()
    counts = result.get_counts()
    
    most_frequent = max(counts, key=counts.get)
    print(f"    {msg}       →       {most_frequent}")

Message Sent → Message Received
------------------------------
    00       →       00
    01       →       01
    10       →       10
    11       →       11
